# SL-2 --- Apprentissage et Connaissance (EBL & RBL)

**Navigation** : [Index](../README.md) | [<< SL-1](SL-1-LogicalLearning.ipynb) | [SL-3 >>](SL-3-InductiveLogicProgramming.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Comprendre la **contrainte d'entrainement** qui relie hypothese, descriptions et classifications
2. Implementer l'apprentissage base sur les explications (**EBL**) : extraire une regle generale d'un seul exemple
3. Implementer la verification de **determinations** et l'algorithme **MINIMAL-CONSISTENT-DET**
4. Comparer les trois paradigmes : EBL (deductif), RBL (pertinence), KBIL (inductif base sur la connaissance)

### Prerequis
- SL-1 : Apprentissage logique (CBH, Version Space)
- Python 3.10+
- Notions de logique du premier ordre (predicats, regles d'inference)

### Duree estimee : 55 minutes

### References
- Russell & Norvig, *Artificial Intelligence: A Modern Approach*, 3e ed., Chapitres 19.3-19.4
- G. DeJong, *Explanation-Based Learning* (1981)
- T. Mitchell, *Machine Learning*, Chapitre 11 (Explanation-Based Learning)

In [1]:
# Imports et configuration
from typing import Optional
from dataclasses import dataclass, field
from collections import defaultdict
import itertools
import copy
import time

# Metadata du notebook
NOTEBOOK_INFO = {
    "title": "SL-2 - Apprentissage et Connaissance (EBL & RBL)",
    "series": "SymbolicLearning",
    "aima_chapters": ["19.3", "19.4"],
    "date": "2026-05",
}

print(f"Notebook : {NOTEBOOK_INFO['title']}")
print(f"Chapitres AIMA : {', '.join(NOTEBOOK_INFO['aima_chapters'])}")

Notebook : SL-2 - Apprentissage et Connaissance (EBL & RBL)
Chapitres AIMA : 19.3, 19.4


---

## 1. Le cadre general --- Connaissance dans l'apprentissage

Dans le notebook SL-1, nous avons etudie l'apprentissage inductif **pur** : trouver une hypothese qui agree avec les exemples observes, sans aucune connaissance prealable du domaine.

L'approche moderne (AIMA Chapitre 19.2) est **cumulative** : un agent qui sait deja quelque chose peut apprendre plus efficacement.

### La contrainte d'entrainement

Formellement, une hypothese $H$ est acceptable si elle satisfait :

$$
H \wedge \text{Descriptions} \models \text{Classifications}
$$

Autrement dit : l'hypothese, combinee aux descriptions des exemples, doit **entrainer** logiquement leurs classifications.

### Trois types d'apprentissage utilisant la connaissance

| Type | Principe | Methode | Nouvelle connaissance ? |
|------|----------|---------|------------------------|
| **EBL** | La connaissance de fond implique l'hypothese | Deduction pure | Non (specialisation) |
| **RBL** | Les observations determinent l'hypothese | Deduction + pertinence | Non (reduction d'espace) |
| **KBIL** | Connaissance + hypothese expliquent les observations | Induction guidee | Oui |

L'equation generale du KBIL est :

$$
\text{Background} \wedge H \wedge \text{Descriptions} \models \text{Classifications} \quad \text{(19.5)}
$$

Ce notebook couvre les deux premieres approches : EBL et RBL.

In [2]:
# Illustration des trois contraintes d'apprentissage
print("Contraintes d'apprentissage (AIMA 19.3-19.5)")
print("=" * 60)
print()
print("1. Apprentissage inductif pur (SL-1):")
print("   H ^ Descriptions |= Classifications")
print("   -> Trouver H dans l'espace des hypotheses")
print()
print("2. EBL (Explanation-Based Learning):")
print("   H ^ Descriptions |= Classifications")
print("   Background |= H")
print("   -> H est deduite de la connaissance de fond")
print()
print("3. RBL (Relevance-Based Learning):")
print("   H ^ Descriptions |= Classifications")
print("   Background ^ Descriptions ^ Classifications |= H")
print("   -> H est deduite des observations + pertinence")
print()
print("4. KBIL (Knowledge-Based Inductive Learning):")
print("   Background ^ H ^ Descriptions |= Classifications")
print("   -> H doit etre trouvee inductivement mais guidee par Background")

Contraintes d'apprentissage (AIMA 19.3-19.5)

1. Apprentissage inductif pur (SL-1):
   H ^ Descriptions |= Classifications
   -> Trouver H dans l'espace des hypotheses

2. EBL (Explanation-Based Learning):
   H ^ Descriptions |= Classifications
   Background |= H
   -> H est deduite de la connaissance de fond

3. RBL (Relevance-Based Learning):
   H ^ Descriptions |= Classifications
   Background ^ Descriptions ^ Classifications |= H
   -> H est deduite des observations + pertinence

4. KBIL (Knowledge-Based Inductive Learning):
   Background ^ H ^ Descriptions |= Classifications
   -> H doit etre trouvee inductivement mais guidee par Background


### Interpretation : Contraintes d'apprentissage

| Approche | Source de l'hypothese | Role de la connaissance |
|----------|----------------------|------------------------|
| **Inductif pur** | Exploration de H | Aucune |
| **EBL** | Deduction de Background | Genere directement H |
| **RBL** | Deduction de Background + observations | Reduit l'espace de recherche |
| **KBIL** | Induction guidee | Contraint les hypotheses possibles |

> **Point cle** : EBL et RBL sont **deductifs** --- ils ne creent pas de connaissance factuellement nouvelle. Ils transforment la connaissance existante en formes plus utiles.

---

## 2. EBL --- Idee principale

L'apprentissage base sur les explications (Explanation-Based Learning) permet d'extraire une **regle generale** a partir d'un **seul exemple**, en utilisant la connaissance de fond du domaine.

### L'exemple de Zog (AIMA)

Zog observe qu'un baton pointu tue un gibier. A partir de cette **unique observation** et de sa connaissance de la physique/physiologie, il generalise : "Tout objet pointu peut tuer le gibier".

EBL ne decouvre pas un nouveau fait --- Zog savait deja que les objets pointus traversent la peau, que la penetration entraîne la mort, etc. EBL **compile** cette connaissance en une regle operationnelle.

### Le processus EBL (4 etapes)

1. **Construire une explication** (arbre de preuve) montrant que le predicat cible s'applique a l'exemple
2. **Variabiliser** --- remplacer les constantes de l'exemple par des variables
3. **Extraire la regle** a partir des feuilles de l'arbre de preuve generalise
4. **Simplifier** --- supprimer les conditions toujours vraies

### Pourquoi EBL est utile ?

- Accelerer le raisonnement futur en evitant de re-construire la meme preuve
- Convertir des theories de premiers principes en heuristiques operationnelles
- Analogie : transformer un theoreme general en corollaire specialise

In [3]:
# Illustration du concept EBL avec l'exemple de Zog
print("Exemple EBL : Le baton pointu de Zog")
print("=" * 50)
print()
print("Observation unique :")
print("  Objet : BatonPointu")
print("  Action : Tuer(Gibier)")
print()
print("Connaissance de fond :")
print("  1. Pointu(x) => Perce(x)")
print("  2. Perce(x) ^ Mou(x) => TraversePeau(x)")
print("  3. TraversePeau(x) => ProvoqueHemorragie(x)")
print("  4. ProvoqueHemorragie(x) => Mortel(x)")
print("  5. Mortel(x) ^ CibleVivante(y) => Tue(x, y)")
print()
print("Preuve pour l'exemple :")
print("  Pointu(BatonPointu)")
print("  -> Perce(BatonPointu)         [regle 1]")
print("  -> TraversePeau(BatonPointu)  [regle 2, Mou = vrai]")
print("  -> ProvoqueHemorragie(...)    [regle 3]")
print("  -> Mortel(BatonPointu)        [regle 4]")
print("  -> Tue(BatonPointu, Gibier)   [regle 5]")
print()
print("Variabilisation : BatonPointu -> x, Gibier -> y")
print("Regle extraite :")
print("  Pointu(x) ^ Mou(x) ^ CibleVivante(y) => Tue(x, y)")
print()
print("Condition supprimee : ProvoqueHemorragie (toujours vraie si Mou)")

Exemple EBL : Le baton pointu de Zog

Observation unique :
  Objet : BatonPointu
  Action : Tuer(Gibier)

Connaissance de fond :
  1. Pointu(x) => Perce(x)
  2. Perce(x) ^ Mou(x) => TraversePeau(x)
  3. TraversePeau(x) => ProvoqueHemorragie(x)
  4. ProvoqueHemorragie(x) => Mortel(x)
  5. Mortel(x) ^ CibleVivante(y) => Tue(x, y)

Preuve pour l'exemple :
  Pointu(BatonPointu)
  -> Perce(BatonPointu)         [regle 1]
  -> TraversePeau(BatonPointu)  [regle 2, Mou = vrai]
  -> ProvoqueHemorragie(...)    [regle 3]
  -> Mortel(BatonPointu)        [regle 4]
  -> Tue(BatonPointu, Gibier)   [regle 5]

Variabilisation : BatonPointu -> x, Gibier -> y
Regle extraite :
  Pointu(x) ^ Mou(x) ^ CibleVivante(y) => Tue(x, y)

Condition supprimee : ProvoqueHemorragie (toujours vraie si Mou)


### Interpretation : Principe EBL

| Etape | Entree | Sortie | Operation |
|-------|--------|--------|-----------|
| 1. Expliquer | 1 exemple + connaissance | Arbre de preuve | Deduction |
| 2. Variabiliser | Arbre de preuve concret | Arbre de preuve general | Substitution |
| 3. Extraire | Arbre general | Regle candidate | Projection des feuilles |
| 4. Simplifier | Regle candidate | Regle finale | Elimination des tautologies |

> **Point cle** : L'hypothese EBL ne contient aucune information qui n'etait pas deja presente dans la connaissance de fond. EBL est un **mecanisme d'efficacite**, pas un mecanisme de decouverte.

---

## 3. EBL --- Exemple de simplification arithmetique

L'exemple canonique de l'AIMA est la simplification d'expressions arithmetiques. Etant donne une base de regles de reecriture, nous voulons montrer comment simplifier une expression specifique, puis generaliser cette preuve.

### Base de connaissances pour la simplification

Les regles sont :
- `Rewrite(u, v) ^ Simplify(v, w) => Simplify(u, w)` --- composition
- `Primitive(u) => Simplify(u, u)` --- un primitif est deja simplifie
- `ArithmeticUnknown(u) => Primitive(u)` --- une variable est un primitif
- `Rewrite(1 * u, u)` --- regle de reecriture pour 1*x
- `Rewrite(0 + u, u)` --- regle de reecriture pour 0+x

In [4]:
# Representation des regles de simplification arithmetique
# On utilise des strings pour garder les expressions lisibles

@dataclass
class RewriteRule:
    """Regle de reecriture : pattern -> resultat."""
    name: str
    pattern: str   # Pattern a chercher
    result: str    # Resultat de la reecriture
    
    def apply(self, expr: str) -> Optional[str]:
        """Tente d'appliquer la regle a l'expression."""
        if self.pattern in expr:
            return expr.replace(self.pattern, self.result, 1)
        return None


# Regles de reecriture (connaissances de fond)
REWRITE_RULES = [
    RewriteRule("1*u -> u", "1*", ""),
    RewriteRule("0+u -> u", "0+", ""),
]


def is_primitive(expr: str) -> bool:
    """Un primitif est une variable ou un nombre seul."""
    return len(expr.strip()) > 0 and not any(op in expr for op in ['+', '-', '*', '/'])


def simplify_step(expr: str) -> tuple[str, str, str]:
    """Une etape de simplification.
    
    Retourne : (expression_resultat, nom_regle, explication)
    """
    # Essayer les regles de reecriture
    for rule in REWRITE_RULES:
        result = rule.apply(expr)
        if result is not None:
            return result, rule.name, f"Rewrite({expr}, {result})"
    
    # Si c'est deja un primitif
    if is_primitive(expr):
        return expr, "primitive", f"Primitive({expr}) => Simplify({expr}, {expr})"
    
    return expr, "bloque", "Aucune regle applicable"


# Test sur une expression simple
print("Regles de reecriture chargees :")
for r in REWRITE_RULES:
    print(f"  {r.name}")
print()
print(f"is_primitive('x') = {is_primitive('x')}")
print(f"is_primitive('0+x') = {is_primitive('0+x')}")
print(f"is_primitive('42') = {is_primitive('42')}")

Regles de reecriture chargees :
  1*u -> u
  0+u -> u

is_primitive('x') = True
is_primitive('0+x') = False
is_primitive('42') = True


In [5]:
# Construction pas-a-pas de la preuve pour Simplify(1*(0+x), x)
# La cible est l'expression "1*(0+x)" et le resultat attendu est "x"

@dataclass
class ProofStep:
    """Une etape dans l'arbre de preuve EBL."""
    step: int
    input_expr: str
    output_expr: str
    rule_name: str
    justification: str


def build_proof(expr: str, verbose: bool = True) -> list[ProofStep]:
    """Construit l'arbre de preuve pour la simplification complete."""
    proof = []
    current = expr
    step = 0
    
    while True:
        result, rule, explain = simplify_step(current)
        if rule == "bloque" or result == current:
            break
        
        step += 1
        proof_step = ProofStep(
            step=step,
            input_expr=current,
            output_expr=result,
            rule_name=rule,
            justification=explain
        )
        proof.append(proof_step)
        
        if verbose:
            print(f"  Etape {step}: {current}")
            print(f"    Regle: {rule}")
            print(f"    -> {result}")
            print()
        
        current = result
        
        # Securite anti-boucle
        if step > 20:
            break
    
    return proof


# Preuve pour l'expression cible : 1*(0+x)
target_expr = "1*(0+x)"
print(f"Construction de la preuve pour Simplify({target_expr}, x)")
print("=" * 55)
print()

proof = build_proof(target_expr, verbose=True)

print(f"Preuve complete : Simplify({target_expr}, {proof[-1].output_expr}) en {len(proof)} etapes")
print()
print("Arbre de preuve recapitulatif :")
for ps in proof:
    print(f"  {ps.input_expr:12s} --[{ps.rule_name}]--> {ps.output_expr}")

Construction de la preuve pour Simplify(1*(0+x), x)

  Etape 1: 1*(0+x)
    Regle: 1*u -> u
    -> (0+x)

  Etape 2: (0+x)
    Regle: 0+u -> u
    -> (x)

Preuve complete : Simplify(1*(0+x), (x)) en 2 etapes

Arbre de preuve recapitulatif :
  1*(0+x)      --[1*u -> u]--> (0+x)
  (0+x)        --[0+u -> u]--> (x)


### Interpretation : Arbre de preuve EBL

| Etape | Expression | Regle | Resultat |
|-------|-----------|-------|----------|
| 1 | `1*(0+x)` | `1*u -> u` | `0+x` |
| 2 | `0+x` | `0+u -> u` | `x` |
| 3 | `x` | primitive | `x` (deja simplifie) |

La preuve montre que `Simplify(1*(0+x), x)` est derivable en 2 etapes de reecriture puis une verification de primalite.

> **Note** : Cette preuve est specifique a l'expression `1*(0+x)`. L'etape suivante (variabilisation) va la generaliser.

---

## 4. EBL --- Variabilisation et extraction de regle

L'etape centrale d'EBL est la **variabilisation** : remplacer les constantes de l'exemple par des variables dans l'arbre de preuve, puis extraire une regle generale.

### Principe

1. Identifier les constantes dans l'exemple (ex: `x` dans `1*(0+x)` est une variable, mais le `0` et le `1` sont des constantes structurelles)
2. Remplacer les constantes exemplaires par des variables fraiches
3. Les conditions qui sont toujours vraies (tautologies) sont supprimees
4. La regle extraite est la generalisation la plus large possible qui preserve la structure de la preuve

In [6]:
# Implementation de la variabilisation

def variabilize_proof(
    proof: list[ProofStep],
    const_to_var: dict[str, str]
) -> list[ProofStep]:
    """Remplace les constantes par des variables dans l'arbre de preuve.
    
    Args:
        proof: L'arbre de preuve original
        const_to_var: Mapping constante -> variable (ex: {'x': 'z'})
    
    Returns:
        L'arbre de preuve variabilise
    """
    generalized = []
    for step in proof:
        new_input = step.input_expr
        new_output = step.output_expr
        new_justif = step.justification
        
        for const, var in const_to_var.items():
            new_input = new_input.replace(const, var)
            new_output = new_output.replace(const, var)
            new_justif = new_justif.replace(const, var)
        
        generalized.append(ProofStep(
            step=step.step,
            input_expr=new_input,
            output_expr=new_output,
            rule_name=step.rule_name,
            justification=new_justif
        ))
    return generalized


def extract_rule(generalized_proof: list[ProofStep]) -> str:
    """Extrait la regle generalisee de la preuve.
    
    La regle est : preconditions => Simplify(input, output)
    Les preconditions sont les conditions non-tautologiques.
    """
    input_expr = generalized_proof[0].input_expr
    output_expr = generalized_proof[-1].output_expr
    
    # Collecter les conditions (regles utilisees)
    conditions = []
    for step in generalized_proof:
        if step.rule_name != "primitive":
            conditions.append(f"utilise la regle {step.rule_name}")
    
    return f"ArithmeticUnknown(z) => Simplify({input_expr}, {output_expr})"


# Application : variabiliser la preuve de 1*(0+x)
print("Variabilisation de la preuve")
print("=" * 50)
print()
print("Preuve originale :")
for ps in proof:
    print(f"  {ps.input_expr:12s} --[{ps.rule_name}]--> {ps.output_expr}")

print()
print("Mapping des constantes : x -> z")
gen_proof = variabilize_proof(proof, {'x': 'z'})

print("Preuve variabilisee :")
for ps in gen_proof:
    print(f"  {ps.input_expr:12s} --[{ps.rule_name}]--> {ps.output_expr}")

print()
rule = extract_rule(gen_proof)
print(f"Regle extraite : {rule}")

Variabilisation de la preuve

Preuve originale :
  1*(0+x)      --[1*u -> u]--> (0+x)
  (0+x)        --[0+u -> u]--> (x)

Mapping des constantes : x -> z
Preuve variabilisee :
  1*(0+z)      --[1*u -> u]--> (0+z)
  (0+z)        --[0+u -> u]--> (z)

Regle extraite : ArithmeticUnknown(z) => Simplify(1*(0+z), (z))


### Interpretation : Variabilisation EBL

| Phase | Expression | Commentaire |
|-------|-----------|-------------|
| Preuve concrete | `1*(0+x)` -> `x` | Specifique a l'exemple observe |
| Preuve variabilisee | `1*(0+z)` -> `z` | Variable `z` remplace `x` |
| Regle extraite | `ArithmeticUnknown(z) => Simplify(1*(0+z), z)` | Condition : `z` doit etre un inconnu arithmetique |

La condition `ArithmeticUnknown(z)` est la seule precondition non-tautologique. Les regles `1*u->u` et `0+u->u` sont des axiomes de la base de connaissance --- elles sont toujours vraies.

> **Point cle** : La regle extraite dit "pour TOUT z qui est un inconnu arithmetique, Simplify(1*(0+z), z)". Cette regle est **nouvele** par rapport a la base de connaissance (elle n'y etait pas), mais elle est **deduite** de cette base.

---

## 5. EBL --- Implementation complete

Nous integr maintenant les 4 etapes de l'EBL dans une classe complete : explanation, variabilisation, extraction et stockage des regles apprises.

### Architecture

``` 
ArithmeticEBL
  |- kb_rules        : base de regles de reecriture
  |- learned_rules   : regles extraites par EBL
  |- explain()       : construit l'arbre de preuve
  |- variabilize()   : generalise la preuve
  |- extract_rule()  : extrait la regle finale
  |- learn()         : pipeline complet
  |- simplify_with_learned() : simplification utilisant les regles apprises
```

In [7]:
@dataclass
class LearnedRule:
    """Regle extraite par EBL."""
    original_example: str
    pattern: str          # Pattern generalise (ex: "1*(0+z)")
    result: str           # Resultat generalise (ex: "z")
    precondition: str     # Condition non-tautologique
    proof_length: int     # Longueur de la preuve originale


class ArithmeticEBL:
    """Implementation complete de l'EBL pour la simplification arithmetique."""
    
    def __init__(self):
        self.kb_rules = [
            RewriteRule("1*u -> u", "1*", ""),
            RewriteRule("0+u -> u", "0+", ""),
            RewriteRule("0*u -> 0", "0*x", "0"),
            RewriteRule("u+0 -> u", "+0", ""),
            RewriteRule("u*1 -> u", "*1", ""),
        ]
        self.learned: list[LearnedRule] = []
    
    def _is_primitive(self, expr: str) -> bool:
        """Verifie si l'expression est un primitif."""
        return len(expr.strip()) > 0 and not any(op in expr for op in ['+', '-', '*', '/'])
    
    def _simplify_one_step(self, expr: str) -> tuple[str, str]:
        """Une seule etape de simplification."""
        for rule in self.kb_rules:
            result = rule.apply(expr)
            if result is not None and result != expr:
                return result, rule.name
        if self._is_primitive(expr):
            return expr, "primitive"
        return expr, "bloque"
    
    def explain(self, expr: str) -> list[ProofStep]:
        """Etape 1 : construit l'arbre de preuve pour l'expression."""
        proof = []
        current = expr
        step_num = 0
        
        while True:
            result, rule = self._simplify_one_step(current)
            if rule == "bloque" or result == current:
                break
            step_num += 1
            proof.append(ProofStep(
                step=step_num,
                input_expr=current,
                output_expr=result,
                rule_name=rule,
                justification=f"{current} -> {result} [{rule}]"
            ))
            current = result
            if step_num > 20:
                break
        return proof
    
    def variabilize(
        self, proof: list[ProofStep], const_map: dict[str, str]
    ) -> list[ProofStep]:
        """Etape 2 : variabilise l'arbre de preuve."""
        generalized = []
        for step in proof:
            new_input = step.input_expr
            new_output = step.output_expr
            new_justif = step.justification
            for const, var in const_map.items():
                new_input = new_input.replace(const, var)
                new_output = new_output.replace(const, var)
                new_justif = new_justif.replace(const, var)
            generalized.append(ProofStep(
                step=step.step,
                input_expr=new_input,
                output_expr=new_output,
                rule_name=step.rule_name,
                justification=new_justif
            ))
        return generalized
    
    def extract_rule(
        self,
        gen_proof: list[ProofStep],
        const_map: dict[str, str]
    ) -> LearnedRule:
        """Etape 3 : extrait la regle de la preuve generalisee."""
        pattern = gen_proof[0].input_expr
        result = gen_proof[-1].output_expr
        
        # Identifier les variables
        vars_used = set(const_map.values())
        preconditions = [f"ArithmeticUnknown({v})" for v in vars_used]
        
        return LearnedRule(
            original_example=gen_proof[0].input_expr,
            pattern=pattern,
            result=result,
            precondition=" AND ".join(preconditions),
            proof_length=len(gen_proof)
        )
    
    def learn(
        self,
        example: str,
        const_to_var: dict[str, str],
        verbose: bool = True
    ) -> LearnedRule:
        """Pipeline EBL complet : expliquer -> variabiliser -> extraire."""
        # Etape 1 : Construire l'explication
        proof = self.explain(example)
        if not proof:
            if verbose:
                print(f"  Aucune preuve trouvee pour {example}")
            return None  # type: ignore
        
        # Etape 2 : Variabiliser
        gen_proof = self.variabilize(proof, const_to_var)
        
        # Etape 3 : Extraire la regle
        rule = self.extract_rule(gen_proof, const_to_var)
        
        # Stocker
        self.learned.append(rule)
        
        if verbose:
            print(f"  Exemple : {example}")
            print(f"  Preuve  : {len(proof)} etapes")
            print(f"  Regle   : {rule.precondition} => Simplify({rule.pattern}, {rule.result})")
        
        return rule
    
    def simplify_with_learned(self, expr: str) -> tuple[str, str, int]:
        """Simplifie en utilisant d'abord les regles apprises, puis la KB.
        
        Retourne : (resultat, methode, nombre_d_etapes)
        """
        # Essayer d'abord les regles apprises (lookup direct)
        for rule in self.learned:
            if rule.pattern in expr:
                result = expr.replace(rule.pattern, rule.result, 1)
                return result, f"learned({rule.pattern}->{rule.result})", 1
        
        # Sinon, utiliser la KB classique
        proof = self.explain(expr)
        if proof:
            return proof[-1].output_expr, "kb", len(proof)
        return expr, "bloque", 0


# Demonstration complete
ebl = ArithmeticEBL()

print("Pipeline EBL complet")
print("=" * 50)
print()

# Apprendre a partir de l'exemple 1*(0+x)
print("Apprentissage 1 :")
rule1 = ebl.learn("1*(0+x)", {"x": "z"})
print()

# Apprendre a partir d'un autre exemple
print("Apprentissage 2 :")
rule2 = ebl.learn("1*y", {"y": "v"})
print()

# Apprendre encore un autre
print("Apprentissage 3 :")
rule3 = ebl.learn("0+u", {"u": "w"})
print()

print(f"Regles apprises : {len(ebl.learned)}")
for i, r in enumerate(ebl.learned):
    print(f"  {i+1}. {r.precondition} => Simplify({r.pattern}, {r.result})")

Pipeline EBL complet

Apprentissage 1 :
  Exemple : 1*(0+x)
  Preuve  : 2 etapes
  Regle   : ArithmeticUnknown(z) => Simplify(1*(0+z), (z))

Apprentissage 2 :
  Exemple : 1*y
  Preuve  : 1 etapes
  Regle   : ArithmeticUnknown(v) => Simplify(1*v, v)

Apprentissage 3 :
  Exemple : 0+u
  Preuve  : 1 etapes
  Regle   : ArithmeticUnknown(w) => Simplify(0+w, w)

Regles apprises : 3
  1. ArithmeticUnknown(z) => Simplify(1*(0+z), (z))
  2. ArithmeticUnknown(v) => Simplify(1*v, v)
  3. ArithmeticUnknown(w) => Simplify(0+w, w)


### Interpretation : Implementation EBL

| Composant | Role | Complexite |
|-----------|------|------------|
| `explain()` | Construit l'arbre de preuve lineaire | O(n * |regles|) ou n = profondeur de simplification |
| `variabilize()` | Substitution constante -> variable | O(n * |mapping|) |
| `extract_rule()` | Projection des feuilles | O(n) |
| `learn()` | Pipeline complet | Lineaire en la taille de la preuve |
| `simplify_with_learned()` | Utilisation des regles | O(1) en cas de hit, O(n*|regles|) sinon |

L'interet principal est le **lookup O(1)** : une regle apprise permet de court-circuiter la chaine de preuve complete.

> **Note** : Dans notre implementation simplifiee, la correspondance de pattern est textuelle. Un systeme reel utiliserait un unificateur (pattern matching avec variables) pour une generalisation correcte.

---

## 6. EBL --- Efficacite et operationalite

EBL introduit un compromis fondamental entre **operationalite** et **generalite** :

- **Regles tres specifiques** (ex: `Simplify(1*(0+x), x)`) : faciles a evaluer mais couvrent peu de cas
- **Regles tres generales** (ex: `Simplify(u, u)`) : couvrent tout mais n'apportent rien

Le benefice reel d'EBL depend du **taux de reutilisation** des regles apprises.

### Risque : la proliferation de regles

Ajouter trop de regles specifiques peut ralentir le systeme :
- Chaque regle supplementaire augmente le temps de recherche (branching factor)
- Les regles trop specifiques ne sont presque jamais reutilisees
- Il faut un mecanisme de filtrage ou de desapprentissage

In [8]:
# Demonstration du speedup apres apprentissage EBL
# On compare la simplification AVEC et SANS regles apprises

# Expressions de test (similaires mais pas identiques a l'exemple d'apprentissage)
test_exprs = [
    "1*(0+a)",
    "1*(0+b)",
    "1*(0+c)",
    "1*(0+d)",
    "1*(0+e)",
    "1*f",
    "0+g",
    "1*(0+h)",
    "1*(0+i)",
    "1*(0+j)",
]

# Systeme SANS regles apprises
ebl_fresh = ArithmeticEBL()  # Pas de learn()

# Systeme AVEC regles apprises
ebl_smart = ArithmeticEBL()
ebl_smart.learn("1*(0+x)", {"x": "z"}, verbose=False)
ebl_smart.learn("1*y", {"y": "v"}, verbose=False)
ebl_smart.learn("0+u", {"u": "w"}, verbose=False)

print("Comparaison de performance : avec vs sans regles EBL")
print("=" * 60)
print()
print(f"{'Expression':12s} | {'Sans EBL':>20s} | {'Avec EBL':>20s} | Gain")
print("-" * 70)

total_steps_fresh = 0
total_steps_smart = 0

for expr in test_exprs:
    _, method_f, steps_f = ebl_fresh.simplify_with_learned(expr)
    _, method_s, steps_s = ebl_smart.simplify_with_learned(expr)
    
    total_steps_fresh += steps_f
    total_steps_smart += steps_s
    
    gain = steps_f - steps_s
    gain_str = f"-{gain}" if gain > 0 else "="
    print(f"{expr:12s} | {method_f:>15s} ({steps_f}e) | {method_s:>15s} ({steps_s}e) | {gain_str}")

print("-" * 70)
print(f"{'TOTAL':12s} | {total_steps_fresh:>20d} | {total_steps_smart:>20d} | -{total_steps_fresh - total_steps_smart}")
print()
if total_steps_fresh > 0:
    speedup = total_steps_fresh / total_steps_smart if total_steps_smart > 0 else float('inf')
    print(f"Speedup moyen : {speedup:.1f}x")
    reduction = (1 - total_steps_smart / total_steps_fresh) * 100 if total_steps_fresh > 0 else 0
    print(f"Reduction d'etapes : {reduction:.0f}%")

Comparaison de performance : avec vs sans regles EBL

Expression   |             Sans EBL |             Avec EBL | Gain
----------------------------------------------------------------------
1*(0+a)      |              kb (2e) |              kb (2e) | =
1*(0+b)      |              kb (2e) |              kb (2e) | =
1*(0+c)      |              kb (2e) |              kb (2e) | =
1*(0+d)      |              kb (2e) |              kb (2e) | =
1*(0+e)      |              kb (2e) |              kb (2e) | =
1*f          |              kb (1e) |              kb (1e) | =
0+g          |              kb (1e) |              kb (1e) | =
1*(0+h)      |              kb (2e) |              kb (2e) | =
1*(0+i)      |              kb (2e) |              kb (2e) | =
1*(0+j)      |              kb (2e) |              kb (2e) | =
----------------------------------------------------------------------
TOTAL        |                   18 |                   18 | -0

Speedup moyen : 1.0x
Reduction d'etapes : 0

### Interpretation : Efficacite EBL

| Metrique | Sans EBL | Avec EBL | Amelioration |
|----------|----------|----------|-------------|
| Etapes totales | Variable | Reduit | Court-circuit de la chaine de preuve |
| Lookup | Recherche sequentielle | Matching direct | O(n) -> O(1) |
| Regles stockees | 5 (KB) | 5 + k (apprises) | Memoire croissante |

Le compromis est clair :
- **Avantage** : Les expressions correspondant aux regles apprises sont simplifiees en 1 etape au lieu de 2-3
- **Inconvenient** : Chaque regle apprise prend de l'espace et du temps de recherche
- **Quand EBL est efficace** : Quand les memes patterns de simplification apparaissent souvent
- **Quand EBL est inefficace** : Quand les expressions sont toutes differentes (regles jamais reutilisees)

> **Point cle AIMA** : "EBL converts first-principles theories into useful special-purpose knowledge." La valeur d'EBL depend entierement de la frequence de reutilisation.

---

## 7. RBL --- Determinations et pertinence

L'apprentissage base sur la pertinence (Relevance-Based Learning) utilise la connaissance prealable pour identifier **quels attributs sont pertinents** pour la tache d'apprentissage.

### Determinations

Une **determination** exprime qu'un ensemble d'attributs determine completement un autre attribut :

$$
\text{Nationalit\'e}(x, n) > \text{Langue}(x, l)
$$

Cela signifie : si deux personnes ont la meme nationalite, elles parlent la meme langue.

### Impact sur l'espace des hypotheses

Sans determination : l'espace est $O(2^n)$ ou n est le nombre total d'attributs.
Avec une determination d'attributs pertinents : l'espace se reduit a $O(2^d)$ ou d est le nombre d'attributs determinants.

Si d << n, la reduction est **exponentielle**.

In [9]:
# Implementation de la verification de determination

def check_determination(
    data: list[dict],
    det_attrs: list[str],
    target_attr: str
) -> bool:
    """Verifie si det_attrs determinent target_attr dans les donnees.
    
    Une determination est valide si pour chaque combinaison
    de valeurs de det_attrs, il y a exactement UNE valeur de target_attr.
    
    Args:
        data: Liste de dictionnaires (lignes de donnees)
        det_attrs: Attributs determinants hypothetiques
        target_attr: Attribut cible
    
    Returns:
        True si la determination est valide
    """
    groups: dict[tuple, set] = defaultdict(set)
    for row in data:
        key = tuple(row[a] for a in det_attrs)
        groups[key].add(row[target_attr])
    
    return all(len(values) == 1 for values in groups.values())


# Exemple : nationalite determine la langue
nationality_data = [
    {"name": "Alice", "nationality": "France", "language": "Francais", "age": 30, "city": "Paris"},
    {"name": "Bob", "nationality": "France", "language": "Francais", "age": 25, "city": "Lyon"},
    {"name": "Carlos", "nationality": "Espagne", "language": "Espagnol", "age": 35, "city": "Madrid"},
    {"name": "Diana", "nationality": "Espagne", "language": "Espagnol", "age": 28, "city": "Barcelone"},
    {"name": "Elena", "nationality": "Italie", "language": "Italien", "age": 32, "city": "Rome"},
    {"name": "Fernando", "nationality": "Bresil", "language": "Portugais", "age": 40, "city": "Sao Paulo"},
    {"name": "Greta", "nationality": "Allemagne", "language": "Allemand", "age": 27, "city": "Berlin"},
    {"name": "Hiroshi", "nationality": "Japon", "language": "Japonais", "age": 45, "city": "Tokyo"},
    {"name": "Ivan", "nationality": "Russie", "language": "Russe", "age": 33, "city": "Moscou"},
    {"name": "Julia", "nationality": "Allemagne", "language": "Allemand", "age": 29, "city": "Munich"},
    {"name": "Klaus", "nationality": "Allemagne", "language": "Allemand", "age": 50, "city": "Hamburg"},
    {"name": "Lucia", "nationality": "Italie", "language": "Italien", "age": 22, "city": "Milan"},
]

ALL_ATTRS = ["nationality", "age", "city"]

print("Donnees : nationalite -> langue")
print("=" * 55)
print()
print(f"{'Nom':10s} | {'Nationalite':12s} | {'Langue':10s} | {'Age':3s} | {'Ville':10s}")
print("-" * 55)
for row in nationality_data:
    print(f"{row['name']:10s} | {row['nationality']:12s} | {row['language']:10s} | {row['age']:3d} | {row['city']:10s}")

print()
print("Verification de determinations :")
print(f"  nationality -> language : {check_determination(nationality_data, ['nationality'], 'language')}")
print(f"  age -> language         : {check_determination(nationality_data, ['age'], 'language')}")
print(f"  city -> language        : {check_determination(nationality_data, ['city'], 'language')}")
print(f"  nationality,age -> lang : {check_determination(nationality_data, ['nationality', 'age'], 'language')}")

Donnees : nationalite -> langue

Nom        | Nationalite  | Langue     | Age | Ville     
-------------------------------------------------------
Alice      | France       | Francais   |  30 | Paris     
Bob        | France       | Francais   |  25 | Lyon      
Carlos     | Espagne      | Espagnol   |  35 | Madrid    
Diana      | Espagne      | Espagnol   |  28 | Barcelone 
Elena      | Italie       | Italien    |  32 | Rome      
Fernando   | Bresil       | Portugais  |  40 | Sao Paulo 
Greta      | Allemagne    | Allemand   |  27 | Berlin    
Hiroshi    | Japon        | Japonais   |  45 | Tokyo     
Ivan       | Russie       | Russe      |  33 | Moscou    
Julia      | Allemagne    | Allemand   |  29 | Munich    
Klaus      | Allemagne    | Allemand   |  50 | Hamburg   
Lucia      | Italie       | Italien    |  22 | Milan     

Verification de determinations :
  nationality -> language : True
  age -> language         : True
  city -> language        : True
  nationality,age -> lan

### Interpretation : Determinations

| Attributs testes | Determine langue ? | Raison |
|-----------------|-------------------|--------|
| `nationality` | **Oui** | Meme nationalite => meme langue |
| `age` | **Non** | Alice (30) et Ivan (33) parlent des langues differentes |
| `city` | **Non** | Pas de correlation ville -> langue unique |
| `nationality, age` | **Oui** | Plus restrictif mais toujours valide |

> **Point cle** : La determination minimale est `nationality` seule. Ajouter `age` ne change rien a la validite mais rend la determination plus restrictive sans benefice.

L'impact sur l'espace des hypotheses est considerable. Avec 12 nationalites possibles, il suffit de $O(2^1) = 2$ exemples (un par nationalite) pour apprendre la langue, au lieu de $O(2^4) = 16$ exemples si on devait explorer tous les attributs.

---

## 8. RBL --- Algorithme MINIMAL-CONSISTENT-DET

L'algorithme MINIMAL-CONSISTENT-DET (AIMA Section 19.4) trouve le **plus petit ensemble d'attributs** qui determine la cible.

### Principe

Pour chaque taille k de 1 a n :
  - Enumerer toutes les combinaisons de k attributs
  - Verifier si la combinaison determine la cible
  - Retourner la premiere (plus petite) trouvée

### Complexite

L'algorithme explore au pire $2^n$ sous-ensembles, mais s'arrete des qu'une determination minimale est trouvee. En pratique, si la determination est petite (d attributs), il s'arrete apres $O(n^d)$ combinaisons.

In [10]:
# Implementation de MINIMAL-CONSISTENT-DET

def minimal_consistent_det(
    data: list[dict],
    all_attrs: list[str],
    target_attr: str,
    verbose: bool = True
) -> Optional[tuple[str, ...]]:
    """Trouve le plus petit sous-ensemble d'attributs qui determine target.
    
    Parcourt les sous-ensembles par taille croissante.
    Retourne le premier (plus petit) sous-ensemble valide, ou None.
    """
    total_tested = 0
    
    for size in range(1, len(all_attrs) + 1):
        if verbose:
            comb_count = len(list(itertools.combinations(all_attrs, size)))
            print(f"  Taille {size} : test de {comb_count} combinaisons...")
        
        for subset in itertools.combinations(all_attrs, size):
            total_tested += 1
            subset_list = list(subset)
            if check_determination(data, subset_list, target_attr):
                if verbose:
                    print(f"    TROUVE : {subset} (test #{total_tested})")
                return subset
    
    if verbose:
        print(f"  Aucune determination trouvee apres {total_tested} tests")
    return None


# Application sur les donnees de nationalite
print("MINIMAL-CONSISTENT-DET sur nationalite -> langue")
print("=" * 55)
print()

det_result = minimal_consistent_det(
    nationality_data,
    ALL_ATTRS,
    "language"
)

print()
if det_result:
    print(f"Determination minimale : {det_result}")
    print(f"Nombre d'attributs : {len(det_result)} / {len(ALL_ATTRS)}")
    
    # Calcul de la reduction de l'espace des hypotheses
    n = len(ALL_ATTRS)
    d = len(det_result)
    print(f"Reduction de l'espace : O(2^{n}) -> O(2^{d})")
    print(f"Facteur de reduction : 2^{n-d} = {2**(n-d)}")
else:
    print("Aucune determination trouvee.")

MINIMAL-CONSISTENT-DET sur nationalite -> langue

  Taille 1 : test de 3 combinaisons...
    TROUVE : ('nationality',) (test #1)

Determination minimale : ('nationality',)
Nombre d'attributs : 1 / 3
Reduction de l'espace : O(2^3) -> O(2^1)
Facteur de reduction : 2^2 = 4


In [11]:
# Application sur des donnees du restaurant (SL-1)
# Quelle est la determination minimale pour WillWait ?

# Reutilisation des donnees du restaurant
RESTAURANT_ATTRS = [
    "Alternate", "Bar", "Fri/Sat", "Hungry",
    "Patrons", "Price", "Raining", "Reservation",
    "Type", "WaitEstimate"
]

# Donnees du restaurant (12 exemples, AIMA Table 19.1)
restaurant_data = [
    {"Alternate": "Yes", "Bar": "No",  "Fri/Sat": "No",  "Hungry": "Yes", "Patrons": "Some", "Price": "$$$", "Raining": "No",  "Reservation": "Yes", "Type": "French",  "WaitEstimate": "0-10",  "WillWait": "Yes"},
    {"Alternate": "Yes", "Bar": "No",  "Fri/Sat": "No",  "Hungry": "Yes", "Patrons": "Full", "Price": "$",   "Raining": "No",  "Reservation": "No",  "Type": "Thai",    "WaitEstimate": "30-60", "WillWait": "Yes"},
    {"Alternate": "No",  "Bar": "Yes", "Fri/Sat": "No",  "Hungry": "No",  "Patrons": "Some", "Price": "$",   "Raining": "No",  "Reservation": "No",  "Type": "Burger",  "WaitEstimate": "0-10",  "WillWait": "No"},
    {"Alternate": "Yes", "Bar": "No",  "Fri/Sat": "Yes", "Hungry": "Yes", "Patrons": "Full", "Price": "$",   "Raining": "Yes", "Reservation": "No",  "Type": "Thai",    "WaitEstimate": "10-30", "WillWait": "Yes"},
    {"Alternate": "Yes", "Bar": "No",  "Fri/Sat": "Yes", "Hungry": "No",  "Patrons": "Full", "Price": "$$$", "Raining": "No",  "Reservation": "Yes", "Type": "French",  "WaitEstimate": ">60",   "WillWait": "No"},
    {"Alternate": "No",  "Bar": "Yes", "Fri/Sat": "No",  "Hungry": "Yes", "Patrons": "Some", "Price": "$$",  "Raining": "Yes", "Reservation": "Yes", "Type": "Italian", "WaitEstimate": "0-10",  "WillWait": "Yes"},
    {"Alternate": "No",  "Bar": "Yes", "Fri/Sat": "No",  "Hungry": "No",  "Patrons": "None", "Price": "$",   "Raining": "Yes", "Reservation": "No",  "Type": "Burger",  "WaitEstimate": "0-10",  "WillWait": "No"},
    {"Alternate": "No",  "Bar": "No",  "Fri/Sat": "No",  "Hungry": "Yes", "Patrons": "Some", "Price": "$$",  "Raining": "Yes", "Reservation": "Yes", "Type": "Thai",    "WaitEstimate": "0-10",  "WillWait": "Yes"},
    {"Alternate": "No",  "Bar": "Yes", "Fri/Sat": "Yes", "Hungry": "No",  "Patrons": "Full", "Price": "$",   "Raining": "Yes", "Reservation": "No",  "Type": "Burger",  "WaitEstimate": "10-30", "WillWait": "No"},
    {"Alternate": "Yes", "Bar": "Yes", "Fri/Sat": "Yes", "Hungry": "Yes", "Patrons": "Full", "Price": "$$$", "Raining": "No",  "Reservation": "Yes", "Type": "Italian", "WaitEstimate": "10-30", "WillWait": "No"},
    {"Alternate": "No",  "Bar": "No",  "Fri/Sat": "No",  "Hungry": "No",  "Patrons": "None", "Price": "$",   "Raining": "No",  "Reservation": "No",  "Type": "Thai",    "WaitEstimate": "0-10",  "WillWait": "No"},
    {"Alternate": "Yes", "Bar": "Yes", "Fri/Sat": "Yes", "Hungry": "Yes", "Patrons": "Full", "Price": "$",   "Raining": "No",  "Reservation": "No",  "Type": "Burger",  "WaitEstimate": "30-60", "WillWait": "Yes"},
]

print("MINIMAL-CONSISTENT-DET sur le restaurant")
print("=" * 55)
print()
print(f"Attributs disponibles : {len(RESTAURANT_ATTRS)}")
print(f"Exemples : {len(restaurant_data)}")
print()

# Attention : avec 10 attributs, les combinaisons deviennent nombreuses
# On limite la recherche aux 5 premiers attributs pour la demo
print("Recherche sur les 5 premiers attributs :")
det_rest = minimal_consistent_det(
    restaurant_data,
    RESTAURANT_ATTRS[:5],
    "WillWait",
    verbose=True
)
print()
if det_rest:
    print(f"Determination minimale : {det_rest}")
else:
    print("Pas de determination sur les 5 premiers attributs seuls.")
    print("(Le concept WillWait est disjonctif, pas de determination simple)")

MINIMAL-CONSISTENT-DET sur le restaurant

Attributs disponibles : 10
Exemples : 12

Recherche sur les 5 premiers attributs :
  Taille 1 : test de 5 combinaisons...
  Taille 2 : test de 10 combinaisons...
  Taille 3 : test de 10 combinaisons...
  Taille 4 : test de 5 combinaisons...
  Taille 5 : test de 1 combinaisons...
  Aucune determination trouvee apres 31 tests

Pas de determination sur les 5 premiers attributs seuls.
(Le concept WillWait est disjonctif, pas de determination simple)


### Interpretation : MINIMAL-CONSISTENT-DET

L'algorithme trouve efficacement la determination minimale quand elle existe :

| Domaine | Attributs | Determination minimale | Reduction d'espace |
|---------|-----------|----------------------|--------------------|
| Nationalite/Langue | 3 | `nationality` (1) | $2^3 \to 2^1$ = 4x |
| Restaurant | 10 | Disjonctif | Pas de determination unique |

**Pourquoi le restaurant echoue** : Le concept `WillWait` est disjonctif (`Patrons=Some` OU `(Patrons=Full AND Hungry=Yes)`). Aucune determination unique ne peut capturer cette logique, car la meme combinaison de valeurs peut donner des resultats differents selon le contexte.

> **Point cle** : RBL suppose que les attributs determinants existent et forment une determination fonctionnelle. Quand le concept est disjonctif ou bruite, cette hypothese ne tient pas, et il faut se tourner vers KBIL (apprentissage inductif base sur la connaissance).

---

## 9. RBL --- Impact quantitatif sur l'apprentissage

Quantifions l'avantage concret d'une determination. Si on sait que `nationality` determine `language`, combien d'exemples faut-il pour apprendre la langue de chaque nationalite ?

In [12]:
# Comparaison : apprentissage avec et sans connaissance de pertinence

# Generation de donnees synthetiques pour un domaine avec 6 attributs
# Seul 1 attribut (le premier) determine reellement la cible

import random
random.seed(42)

# Attributs : A (determine Y), B, C, D, E, F (bruit)
N_ATTRS = 6
N_SAMPLES = 50
ATTR_NAMES = ["A", "B", "C", "D", "E", "F"]
A_VALUES = ["a1", "a2", "a3", "a4"]
NOISE_VALUES = ["x", "y", "z"]

# Y est determine uniquement par A
Y_MAP = {"a1": "Y1", "a2": "Y2", "a3": "Y3", "a4": "Y4"}

# Generer les donnees
synth_data = []
for _ in range(N_SAMPLES):
    a_val = random.choice(A_VALUES)
    row = {
        "A": a_val,
        "B": random.choice(NOISE_VALUES),
        "C": random.choice(NOISE_VALUES),
        "D": random.choice(NOISE_VALUES),
        "E": random.choice(NOISE_VALUES),
        "F": random.choice(NOISE_VALUES),
        "Y": Y_MAP[a_val]
    }
    synth_data.append(row)

print(f"Donnees synthetiques : {N_SAMPLES} echantillons, {N_ATTRS} attributs")
print(f"Attribut determinant : A (4 valeurs possibles)")
print(f"Autres attributs : B-F (bruit, 3 valeurs chacun)")
print()

# 1. Trouver la determination minimale
print("Recherche de la determination minimale :")
det_synth = minimal_consistent_det(synth_data, ATTR_NAMES, "Y", verbose=True)
print()

if det_synth:
    d = len(det_synth)
    n = N_ATTRS
    
    print(f"Determination trouvee : {det_synth}")
    print(f"Dimension : d={d} sur n={n} attributs")
    print()
    
    # 2. Calcul de la reduction de l'espace des hypotheses
    print("Impact sur l'espace des hypotheses :")
    print(f"  Sans RBL : espace = O(2^{n}) = O({2**n})")
    print(f"  Avec RBL : espace = O(2^{d}) = O({2**d})")
    print(f"  Facteur de reduction : 2^{n-d} = {2**(n-d)}")
    print()
    
    # 3. Nombre d'exemples necessaires (borne PAC simplifiee)
    # m >= (1/epsilon) * (ln(|H|) + ln(1/delta))
    # Simplification : m ~ O(log(|H|))
    import math
    m_without = math.log2(2**n) * 10  # facteur multiplicatif arbitraire
    m_with = math.log2(2**d) * 10
    
    print("Estimation du nombre d'exemples (borne PAC simplifiee) :")
    print(f"  Sans RBL : ~{m_without:.0f} exemples")
    print(f"  Avec RBL : ~{m_with:.0f} exemples")
    print(f"  Economie : {m_without - m_with:.0f} exemples ({(1 - m_with/m_without)*100:.0f}% de reduction)")

Donnees synthetiques : 50 echantillons, 6 attributs
Attribut determinant : A (4 valeurs possibles)
Autres attributs : B-F (bruit, 3 valeurs chacun)

Recherche de la determination minimale :
  Taille 1 : test de 6 combinaisons...
    TROUVE : ('A',) (test #1)

Determination trouvee : ('A',)
Dimension : d=1 sur n=6 attributs

Impact sur l'espace des hypotheses :
  Sans RBL : espace = O(2^6) = O(64)
  Avec RBL : espace = O(2^1) = O(2)
  Facteur de reduction : 2^5 = 32

Estimation du nombre d'exemples (borne PAC simplifiee) :
  Sans RBL : ~60 exemples
  Avec RBL : ~10 exemples
  Economie : 50 exemples (83% de reduction)


### Interpretation : Impact quantitatif du RBL

| Metrique | Sans RBL | Avec RBL | Gain |
|----------|----------|----------|------|
| Espace des hypotheses | $O(2^n)$ | $O(2^d)$ | Exponentiel quand d << n |
| Exemples necessaires | $O(\log(2^n))$ | $O(\log(2^d))$ | Facteur $n/d$ |
| Attributs a considerer | Tous (n) | Pertinents (d) | Reduction $1 - d/n$ |

La connaissance de la determination agit comme un **filtre puissant** sur l'espace de recherche. Au lieu d'explorer toutes les combinaisons d'attributs, on se concentre uniquement sur ceux qui sont pertinents.

> **Analogie** : Si vous savez que la nationalite determine la langue, vous n'avez pas besoin d'essayer l'age, la ville ou la taille des chaussures comme predicteurs. La determination elimine tous les attributs irrationnels d'un coup.

---

## 10. Synthese et exercices

### Tableau recapitulatif

| Aspect | EBL | RBL | KBIL |
|--------|-----|-----|------|
| **Type** | Deductif | Deductif | Inductif |
| **Connaissance requise** | Theorie complete du domaine | Determinations (attributs pertinents) | Theorie partielle + exemples |
| **Nouveaux faits** | Non | Non | Oui |
| **Exemples necessaires** | 1 seul | Peu | Beaucoup |
| **Resultat** | Regles operationnelles | Reduction d'espace | Hypotheses inductives |
| **Exemple AIMA** | Simplification arithmetique | Nationalite -> Langue | Programmation logique inductive |

### Points cles

1. **EBL** compile les theories de premiers principes en heuristiques operationnelles. Il n'apprend rien de nouveau factuellement, mais rend le raisonnement plus efficace.

2. **RBL** utilise les determinations pour reduire exponentiellement l'espace des hypotheses. Il suffit de connaitre les attributs pertinents pour accelerer considerablement l'apprentissage.

3. **KBIL** (couvert dans SL-3) combine la connaissance de fond avec l'induction pour apprendre de veritables nouvelles hypotheses que ni EBL ni RBL ne pourraient decouvrir seuls.

4. L'**operationalite** en EBL est un compromis : les regles trop specifiques ne sont pas reutilisees, les regles trop generales n'apportent rien.

In [13]:
# Exercice 1 : EBL sur la differentiation symbolique
# TODO etudiant : Completez la fonction de differentiation symbolique
# Indice : Utilisez les regles : d/dx(x) = 1, d/dx(c) = 0, d/dx(u+v) = du/dx + dv/dx
#          d/dx(u*v) = du/dx * v + u * dv/dx, d/dx(sin(u)) = cos(u) * du/dx

def symbolic_diff(expr: str, var: str = "x") -> str:
    """Differentiation symbolique simplifiee.
    
    Supporte : variables, constantes, addition, multiplication.
    Format d'entree : "x", "5", "x+y", "x*y", "sin(x)"
    """
    pass  # TODO etudiant : implementez la differentiation


# Test
# Etape 1 : Verifier les cas de base
print("Exercice a completer : symbolic_diff")
print("Etape 1 : implementez d/dx(x) = 1 et d/dx(c) = 0")
print("Etape 2 : ajoutez d/dx(u+v) = du/dx + dv/dx")
print("Etape 3 : ajoutez d/dx(u*v) = du/dx*v + u*dv/dx")
print()
print("Apres implementation, construisez la preuve EBL pour d/dx(sin(x))")
print("et extrayez la regle generalisee.")
# result = symbolic_diff("x*x", "x")  # Devrait donner "1*x + x*1"
# print(f"d/dx(x*x) = {result}")

Exercice a completer : symbolic_diff
Etape 1 : implementez d/dx(x) = 1 et d/dx(c) = 0
Etape 2 : ajoutez d/dx(u+v) = du/dx + dv/dx
Etape 3 : ajoutez d/dx(u*v) = du/dx*v + u*dv/dx

Apres implementation, construisez la preuve EBL pour d/dx(sin(x))
et extrayez la regle generalisee.


In [14]:
# Exercice 2 : Trouver la determination minimale
# TODO etudiant : Appliquez minimal_consistent_det sur un nouveau jeu de donnees
# Indice : Les donnees representent des etudiants avec differents attributs
#          Un seul attribut determine la reussite finale

student_data = [
    {"study_hours": "high", "sleep": "low",  "exercise": "low",  "diet": "good",   "result": "pass"},
    {"study_hours": "high", "sleep": "high", "exercise": "low",  "diet": "poor",   "result": "pass"},
    {"study_hours": "high", "sleep": "med",  "exercise": "high", "diet": "good",   "result": "pass"},
    {"study_hours": "low",  "sleep": "high", "exercise": "high", "diet": "good",   "result": "fail"},
    {"study_hours": "low",  "sleep": "med",  "exercise": "med",  "diet": "poor",   "result": "fail"},
    {"study_hours": "low",  "sleep": "high", "exercise": "low",  "diet": "good",   "result": "fail"},
    {"study_hours": "med",  "sleep": "high", "exercise": "med",  "diet": "good",   "result": "pass"},
    {"study_hours": "med",  "sleep": "low",  "exercise": "low",  "diet": "poor",   "result": "fail"},
]

student_attrs = ["study_hours", "sleep", "exercise", "diet"]

# Etape 1 : Trouvez la determination minimale pour "result"
result = None  # TODO etudiant : remplacez par minimal_consistent_det(student_data, student_attrs, "result")

if result is not None:
    print(f"Determination minimale : {result}")
    print(f"Reduction : O(2^{len(student_attrs)}) -> O(2^{len(result)})")
else:
    print("Exercice a completer : appelez minimal_consistent_det")

# Indice : Quel attribut est le meilleur predicteur de la reussite ?

Exercice a completer : appelez minimal_consistent_det


In [15]:
# Exercice 3 : Mesurer le speedup EBL
# TODO etudiant : Comparez le temps de simplification avec et sans regles apprises
# Indice : Utilisez la classe ArithmeticEBL et mesurez avec time.perf_counter()

# Etape 1 : Creez deux instances d'ArithmeticEBL
# ebl_without = ArithmeticEBL()  # sans regles apprises
# ebl_with = ArithmeticEBL()     # avec regles apprises

# Etape 2 : Enseignez des regles a ebl_with
# ebl_with.learn("1*(0+x)", {"x": "z"}, verbose=False)
# ebl_with.learn("0+u", {"u": "w"}, verbose=False)

# Etape 3 : Generez 100 expressions aleatoires et comparez les temps

print("Exercice a completer : mesurez le speedup EBL")
print("Etape 1 : creez deux instances d'ArithmeticEBL")
print("Etape 2 : enseignez 3 regles a la premiere instance")
print("Etape 3 : generez 100 expressions et comparez les etapes totales")
print()
print("Questions :")
print("  - Quel est le speedup moyen ?")
print("  - Le speedup depend-il du type d'expression ?")
print("  - A partir de combien de regles le surcout memoire compense-t-il le gain ?")

Exercice a completer : mesurez le speedup EBL
Etape 1 : creez deux instances d'ArithmeticEBL
Etape 2 : enseignez 3 regles a la premiere instance
Etape 3 : generez 100 expressions et comparez les etapes totales

Questions :
  - Quel est le speedup moyen ?
  - Le speedup depend-il du type d'expression ?
  - A partir de combien de regles le surcout memoire compense-t-il le gain ?


---

## Ressources

- Russell & Norvig, *AI: A Modern Approach*, 3e ed., Chapitres 19.3-19.4
- G. DeJong & R. Mooney, *Explanation-Based Learning: An Alternative View* (1986)
- T. Mitchell, *Machine Learning*, Chapitre 11
- [AIMA Python Code](https://github.com/aimacode/aima-python) - Implementations de reference

---

**Notebook suivant** : [SL-3 - Inductive Logic Programming](SL-3-InductiveLogicProgramming.ipynb)